# Batch Scoring — Outage Prediction

Scores every row with the champion **registered model** from the MLflow registry.

**Reads:** `gold_ml_features`, `gold_ml_scoring_meta`, MLflow registry  **Writes:** `gold_ml_predictions`, `gold_ml_summary`

In [ ]:
import json
import mlflow
import mlflow.lightgbm
import mlflow.sklearn
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, avg, count,
    sum as spark_sum, round as spark_round
)

spark = SparkSession.builder.getOrCreate()

# Load champion registered model + scoring metadata (flavor-aware: predict_proba)
meta = json.loads(spark.read.table('gold_ml_scoring_meta').collect()[0]['meta_json'])
champion_name = meta['champion_model']
champion_flavor = meta['champion_flavor']
feature_cols = meta['feature_cols']
numeric_features = meta['numeric_features']
cat_cols = meta['cat_cols']
category_maps = meta['category_maps']

uri = f'models:/{champion_name}/latest'
model = mlflow.lightgbm.load_model(uri) if champion_flavor == 'lightgbm' else mlflow.sklearn.load_model(uri)
df = spark.read.table('gold_ml_features')
print(f'Loaded registered model: {champion_name}')
print(f'Scoring {df.count():,} feature rows')

In [ ]:
# Prepare features exactly as in training: named float64 pandas columns
pdf = df.toPandas()
for c in numeric_features:
    pdf[c] = pd.to_numeric(pdf[c], errors='coerce').fillna(0.0)
for c in cat_cols:
    pdf[f'{c}_idx'] = pdf[c].astype(str).map(category_maps[c]).fillna(-1).astype('float64')

X_all = pdf[feature_cols].astype('float64')
pdf['probability_positive'] = model.predict_proba(X_all)[:, 1]
pdf['prediction'] = model.predict(X_all).astype('int64')
print(f'Scored {len(pdf):,} rows with {champion_name}')

In [ ]:
# Score every row and derive outage probability + risk level
scored = spark.createDataFrame(pdf)


predictions = (
    scored
    .withColumn('outage_probability', spark_round(col('probability_positive'), 4))
    .withColumn('predicted_outage', col('prediction').cast('int'))
    .withColumn('risk_level',
        when(col('outage_probability') > 0.8, 'critical')
        .when(col('outage_probability') > 0.6, 'high')
        .when(col('outage_probability') > 0.4, 'medium')
        .otherwise('low')
    )
    .withColumn('scored_at', current_timestamp())
    .select(
        'substation_id', 'region', 'sensor_date',
        'avg_voltage', 'avg_frequency', 'avg_power_factor', 'avg_load', 'avg_temp',
        'voltage_deviation', 'freq_deviation',
        'had_outage', 'predicted_outage',
        'outage_probability', 'risk_level',
        'scored_at'
    )
)

predictions.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_predictions')
print(f'Predictions written: {predictions.count()} rows')
predictions.groupBy('risk_level').count().orderBy('count', ascending=False).show()

In [ ]:
# Per-substation risk summary
summary = (
    predictions
    .groupBy('substation_id', 'region')
    .agg(
        spark_round(avg('outage_probability'), 4).alias('avg_outage_risk'),
        spark_sum('predicted_outage').alias('predicted_outage_days'),
        count('*').alias('total_days'),
        spark_round(avg('voltage_deviation'), 2).alias('avg_voltage_deviation'),
        spark_round(avg('freq_deviation'), 4).alias('avg_freq_deviation'),
        spark_round(avg('avg_load'), 2).alias('avg_daily_load'),
    )
    .withColumn('outage_rate', spark_round(col('predicted_outage_days') / col('total_days') * 100, 1))
    .withColumn('overall_risk',
        when(col('avg_outage_risk') > 0.6, 'high')
        .when(col('avg_outage_risk') > 0.3, 'medium')
        .otherwise('low')
    )
    .withColumn('summary_timestamp', current_timestamp())
    .orderBy(col('avg_outage_risk').desc())
)

summary.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_summary')
print(f'Substation risk summary: {summary.count()} substations')
summary.show(10, truncate=False)